# 산학프로젝트
**제조 공정 품질 불량 예측**

# 06_Final_Model

- 최종 모델을 전체 데이터로 학습해 배포용으로 저장
- 평가 프로토콜: 없음. 성능 수치는 04_6 반복 CV, 운영점은 05에서 확정한 값을 그대로 기록
- 주의: 학습과 저장만 하는 노트북. 전체 데이터로 학습했으므로 여기서 성능을 재측정하면 안 됨

## import

In [1]:
import json
import os
import sys
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT=Path.cwd().parent if Path.cwd().name=='data' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'scikit-learn {sklearn.__version__}')

scikit-learn 1.9.0


## 경로설정

In [2]:
if Path.cwd().name!='data':
    os.chdir('data')

## 데이터 불러오기

In [3]:
# 02_EDA에서 저장한 전처리 데이터
# - 기록 오류 보정, 중복 제거, CN7·RG3 한정, PassOrFail 인코딩(양품 0 / 불량 1), Part 파생까지 끝난 상태
model_data=pd.read_csv('labeled_modeling.csv', index_col=0)

print(f'{model_data.shape[0]}행 {model_data.shape[1]}컬럼 / 불량 {int(model_data["PassOrFail"].sum())}건')

5230행 30컬럼 / 불량 60건


## 데이터 전처리

- 03 ~ 05와 동일한 전처리
- 순서와 컬럼 구성이 달라지면 저장한 모델을 쓸 수 없으므로 그대로 유지

In [4]:
# 식별 컬럼 제외
drop_columns=['TimeStamp', 'PART_FACT_PLAN_DATE', 'PART_FACT_SERIAL', 'PART_NAME', 'EQUIP_CD']

final_df=model_data.drop(columns=drop_columns)

X_part=final_df.drop(columns='PassOrFail')
Y=final_df['PassOrFail']

# 더미변수 생성
X_part=pd.get_dummies(data=X_part, columns=['Part'], drop_first=True, dtype=int)

print(f'학습 데이터 {X_part.shape}, 불량 {Y.sum()}건 ({Y.mean():.4%})')

학습 데이터 (5230, 26), 불량 60건 (1.1472%)


## 최종 모델 학습

- 04_6 반복 CV에서 AP 1위, 05에서 검사 물량 전 구간이 안정적이었던 RandomForest Balanced 사용
- 파라미터는 04_2 Grid Search가 선택한 값이며, 04_6이 이 파라미터로 성능을 측정
- 오토인코더 복원 오차는 04_6 결과에 따라 미포함
- 검증 없음. 전체 5230건을 모두 학습에 사용

In [5]:
final_model=Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(n_estimators=300, max_depth=20, min_samples_split=2, min_samples_leaf=4,
                                  class_weight='balanced', n_jobs=-1, random_state=0)),
])

final_model.fit(X_part, Y)
print(final_model)

Pipeline(steps=[('scaler', StandardScaler()),
                ('rf',
                 RandomForestClassifier(class_weight='balanced', max_depth=20,
                                        min_samples_leaf=4, n_estimators=300,
                                        n_jobs=-1, random_state=0))])


In [6]:
# 학습 데이터 예측은 성능 확인용이 아니라 파이프라인 동작 확인용입니다.
sample_proba=final_model.predict_proba(X_part.head(5))[:, 1]
print('앞 5건 불량확률:', np.round(sample_proba, 4))

importance=pd.Series(final_model.named_steps['rf'].feature_importances_, index=X_part.columns)
importance.sort_values(ascending=False).head(10).round(4)

앞 5건 불량확률: [0.1871 0.2238 0.1131 0.2056 0.3669]


Plasticizing_Position     0.1136
Max_Back_Pressure         0.0738
Mold_Temperature_4        0.0691
Clamp_Close_Time          0.0599
Mold_Temperature_3        0.0550
Max_Injection_Pressure    0.0535
Max_Injection_Speed       0.0522
Average_Back_Pressure     0.0489
Part_RG3LH                0.0444
Plasticizing_Time         0.0438
dtype: float64

## 운영 기준 기록

- 05에서 확정한 검사 물량과 Threshold를 함께 저장
- 성능 수치는 04_6 반복 CV 결과이며 이 노트북에서 재계산하지 않음

In [7]:
operating_table=pd.read_csv('05_Operating_Point_threshold.csv')
operating_table=operating_table[operating_table['Model']=='RandomForest Balanced']

recall_table=pd.read_csv('05_Operating_Point_by_ratio.csv')
recall_table=recall_table[recall_table['Model']=='RandomForest Balanced']

operating_points=recall_table.merge(operating_table[['검사 비율', 'Threshold mean', 'Threshold std']], on='검사 비율')
operating_points=operating_points[['검사 비율', '검사 수', 'Threshold mean', 'Threshold std',
                                   'Recall@k 평균', 'Recall@k 표준편차', 'Precision@k 평균', 'Lift 평균']]
operating_points.round(4)

,검사 비율,검사 수,Threshold mean,Threshold std,Recall@k 평균,Recall@k 표준편차,Precision@k 평균,Lift 평균
0,0.01,52,0.6386,0.0124,0.3222,0.0347,0.3718,32.4081
1,0.02,105,0.4969,0.0058,0.4056,0.0419,0.2317,20.2005
2,0.03,157,0.4215,0.0038,0.4500,0.0333,0.1720,14.9904
3,0.05,262,0.3160,0.0044,0.5333,0.0441,0.1221,10.6463
4,0.07,366,0.2495,0.0065,0.6833,0.0289,0.1120,9.7646
5,0.10,523,0.1627,0.0004,0.7889,0.0096,0.0905,7.8889
6,0.15,784,0.0794,0.0013,0.8611,0.0255,0.0659,5.7444
7,0.20,1046,0.0349,0.0017,0.9389,0.0255,0.0539,4.6944


## 모델 저장

- `dashboard/plan.md`의 "5. 모델 적용 계획"이 요구하는 항목을 모두 저장
- 전처리(StandardScaler)는 Pipeline 안에 있으므로 예측 시 원본 값을 그대로 입력
- SMOTE는 최종 모델에 없음

In [8]:
MODEL_DIR=PROJECT_ROOT/'models'
MODEL_DIR.mkdir(exist_ok=True)

MODEL_VERSION='v1.1.0'

model_path=MODEL_DIR/'final_rf_part_balanced.pkl'
joblib.dump(final_model, model_path)

feature_path=MODEL_DIR/'final_feature_columns.pkl'
joblib.dump(list(X_part.columns), feature_path)

metadata={
    'model_version': MODEL_VERSION,
    'created_at': datetime.now().isoformat(timespec='seconds'),
    'model_file': model_path.name,
    'feature_file': feature_path.name,
    'estimator': 'Pipeline(StandardScaler, RandomForestClassifier)',
    'hyperparameters': {
        'n_estimators': 300,
        'max_depth': 20,
        'min_samples_split': 2,
        'min_samples_leaf': 4,
        'class_weight': 'balanced',
        'random_state': 0,
    },
    'training_data': {
        'source': 'labeled_modeling.csv (02_EDA 전처리 결과)',
        'rows': int(len(X_part)),
        'defects': int(Y.sum()),
        'defect_rate': float(Y.mean()),
        'part_names': ["CN7 W/S SIDE MLD'G LH", "CN7 W/S SIDE MLD'G RH",
                       "RG3 MOLD'G W/SHLD, LH", "RG3 MOLD'G W/SHLD, RH"],
    },
    'feature_columns': list(X_part.columns),
    'categorical_encoding': {
        'source_column': 'PART_NAME',
        'derived_column': 'Part',
        'rule': "PART_NAME[:3] + PART_NAME[-2:]",
        'dummy_columns': ['Part_CN7RH', 'Part_RG3LH', 'Part_RG3RH'],
        'baseline_category': 'CN7LH',
        'note': 'drop_first=True 이므로 CN7LH는 더미 3개가 모두 0인 상태로 표현됨',
    },
    'recommended_threshold': 0.163,
    'recommended_inspect_ratio': 0.10,
    'performance': {
        'protocol': '5-fold x 3반복 (04_6_AutoEncoder_CV_Validation)',
        'average_precision_mean': 0.3931,
        'average_precision_std': 0.1107,
        'note': '전체 5230건 기준 반복 CV 결과이며, 이 모델은 같은 데이터 전체로 재학습한 것임',
    },
    'operating_points': json.loads(operating_points.to_json(orient='records', force_ascii=False)),
    'data_corrections': {
        'average_screw_rpm': '100 초과 값은 10으로 나눔 (mm/s 기준, Average <= Max 위반 3021건 -> 0건)',
        'back_pressure': '행별로 Average=min, Max=max로 재구성 (위반 5212건 -> 0건)',
        'dropped_columns': ['Clamp_Open_Position'],
        'note': '02_EDA에서 적용한 규칙. 예측 서버는 dashboard/backend/app/corrections.py에 같은 규칙을 둠',
    },
    'excluded': {
        'autoencoder_recon_error': '04_6 짝비교에서 운영 구간 이득이 없어 미적용',
        'smote': '최종 모델은 class_weight=balanced 사용',
        'pca': '04_3에서 AP 0.05~0.20으로 하락하여 기각',
        'feature_selection': '04_4에서 AP 하락하여 기각',
    },
    'library_versions': {'scikit-learn': sklearn.__version__},
}

metadata_path=MODEL_DIR/'final_model_meta.json'
metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')

for path in [model_path, feature_path, metadata_path]:
    print(f'{path.name} ({path.stat().st_size/1024**2:.2f} MB)')

final_rf_part_balanced.pkl (2.65 MB)
final_feature_columns.pkl (0.00 MB)
final_model_meta.json (0.01 MB)


## 저장 결과 확인

- 저장한 파일만으로 예측이 되는지 확인
- FastAPI 예측 서버가 사용할 순서와 동일

In [9]:
loaded_model=joblib.load(MODEL_DIR/'final_rf_part_balanced.pkl')
loaded_columns=joblib.load(MODEL_DIR/'final_feature_columns.pkl')
loaded_meta=json.loads((MODEL_DIR/'final_model_meta.json').read_text(encoding='utf-8'))

# 예측 예시: 원본 형태의 새 데이터 3건
new_rows=X_part.sample(3, random_state=0)

proba=loaded_model.predict_proba(new_rows[loaded_columns])[:, 1]
threshold=loaded_meta['recommended_threshold']

result=pd.DataFrame({
    '불량확률': proba.round(4),
    '판정': np.where(proba>=threshold, '불량 위험', '정상'),
})
print(f"모델 버전 {loaded_meta['model_version']} / Threshold {threshold}")
result

모델 버전 v1.1.0 / Threshold 0.163


,불량확률,판정
0,0.0000,정상
1,0.0128,정상
2,0.3149,불량 위험


In [10]:
# 저장한 컬럼 순서와 학습 컬럼 순서가 같은지 확인
assert loaded_columns==list(X_part.columns), '컬럼 순서 불일치'

# 전체 데이터에 Threshold를 적용했을 때 알람 비율 확인 (성능 측정이 아니라 물량 확인)
all_proba=loaded_model.predict_proba(X_part[loaded_columns])[:, 1]
alarm_ratio=(all_proba>=threshold).mean()
print(f'컬럼 순서 일치, Threshold {threshold} 적용 시 알람 비율 {alarm_ratio:.2%}')
print('참고: 학습 데이터에 대한 예측이므로 운영 시 알람 비율과 정확히 같지는 않습니다. 목표 물량은 10%입니다.')

컬럼 순서 일치, Threshold 0.163 적용 시 알람 비율 9.87%
참고: 학습 데이터에 대한 예측이므로 운영 시 알람 비율과 정확히 같지는 않습니다. 목표 물량은 10%입니다.
